In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
dress_model = tf.keras.models.load_model("/content/drive/MyDrive/pr/dress_model.keras")
fabric_model = tf.keras.models.load_model("/content/drive/MyDrive/pr/fabric_model.keras")
defect_model = tf.keras.models.load_model("/content/drive/MyDrive/pr/defect_model.keras")

In [30]:
dress_classes = ['dhoti', 'frock', 'salwar', 'saree', 'shirt']

fabric_classes = [
    'acrylic','artificial_fur','artificial_leather','blended','chenille',
    'corduroy','cotton','crepe','denim','felt','fleece','leather','linen',
    'nylon','polyester','satin','silk','suede','terrycloth',
    'unclassified','velvet','viscose','wool'
]

defect_classes = ['hole', 'horizontal', 'lines', 'no_defect', 'vertical']

In [31]:
def zoom_image(img, zoom=1.5):

    h, w, _ = img.shape

    new_h, new_w = int(h / zoom), int(w / zoom)

    start_h = (h - new_h) // 2
    start_w = (w - new_w) // 2

    cropped = img[start_h:start_h+new_h, start_w:start_w+new_w]

    cropped = tf.image.resize(cropped, (224, 224))

    return cropped

In [32]:
def preprocess_dress(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img = image.img_to_array(img)
    img = img / 255.0
    return np.expand_dims(img, axis=0)

In [33]:
def preprocess_fabric(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img = image.img_to_array(img)

    img = zoom_image(img, zoom=1.8)
    img = img / 255.0

    return np.expand_dims(img, axis=0)

In [34]:
def preprocess_defect(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img = image.img_to_array(img)

    img = zoom_image(img, zoom=2.0)
    img = img / 255.0

    return np.expand_dims(img, axis=0)

In [35]:
def predict_all(img_path):

    dress_img = preprocess_dress(img_path)
    dress_pred = dress_model.predict(dress_img)
    dress = dress_classes[np.argmax(dress_pred)]

    fabric_img = preprocess_fabric(img_path)
    fabric_pred = fabric_model.predict(fabric_img)
    fabric = fabric_classes[np.argmax(fabric_pred)]


    defect_img = preprocess_defect(img_path)
    defect_pred = defect_model.predict(defect_img)

    defect_idx = np.argmax(defect_pred)
    defect_conf = np.max(defect_pred) * 100
    defect = defect_classes[defect_idx]

    print("\n===== FINAL RESULT =====")
    print("Dress Type:", dress)
    print("Fabric Type:", fabric)

    if defect_conf < 60:
        print("Defect: NO DEFECT (Good Quality)")
    else:
        print("Defect Type:", defect)
        print("Confidence:", f"{defect_conf:.2f}%")

        if defect == "Hole":
            print(" Reason: Missing fabric area detected")
        elif defect == "Horizontal":
            print(" Reason: Horizontal crack/scratch detected")
        elif defect == "Vertical":
            print(" Reason: Vertical fiber damage detected")
        elif defect == "Lines":
            print("Reason: Multiple scratch marks detected")

In [42]:
from google.colab import files

uploaded = files.upload()

Saving 32b2420e4be4b3a6dd043ba0ec42f2d7.jpg to 32b2420e4be4b3a6dd043ba0ec42f2d7 (1).jpg


In [43]:
img_path = list(uploaded.keys())[0]
predict_all(img_path)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step

===== FINAL RESULT =====
Dress Type: saree
Fabric Type: cotton
Defect Type: no_defect
Confidence: 97.77%


'32b2420e4be4b3a6dd043ba0ec42f2d7 (1).jpg'  'OIP 1 (2).jpg'    'OIP (2).jpg'
 32b2420e4be4b3a6dd043ba0ec42f2d7.jpg	    'OIP 1.jpg'        'OIP (3).jpg'
 drive					    'OIP (1).jpg'       OIP.jpg
'OIP 1 (1).jpg'				    'OIP (2) (1).jpg'   sample_data
